# Colab: Run All Pending Virchow Tests

This notebook runs four aligned evaluations:

1. CAMELYON17-trained model -> CAMELYON17 reserved test
2. CAMELYON17-trained model -> PCam reserved test
3. PCam-trained model -> CAMELYON17 reserved test
4. PCam-trained model -> PCam reserved test

It uses `scripts/evaluate_virchow_preprocessed_test_colab.py` and writes each test to its own output folder to avoid overwriting metrics.

Evaluation runs with Colab’s **`python` on `PATH`** (like `!python` in the beginner notebook), **`python -u`** for live logs, **`--batch-size 320`**, and **`--num-workers 0`**.

**Resume after disconnect:** Run setup cells again; each test is its own cell. Completed runs skip when `test_metrics.json` exists under `EVAL_OUTPUT_ROOT` on Drive (then synced from `/content`).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!nvidia-smi

## Repo Setup
If your repo is already available in Colab/Drive, skip clone and set `REPO_DIR` accordingly in the next cell.

In [ ]:
%cd /content
!git clone https://github.com/LeenRayess/GP_ECG.git GP_ECG
%cd /content/GP_ECG

In [ ]:
# ===================== EDIT THIS CELL =====================
REPO_DIR = '/content/GP_ECG'

# Training run directories (contain model_best.pt, run_config.json, temperature_fit.json)
RUN_DIR_CAM17_TRAINED = '/content/drive/MyDrive/GP_ECG_RUNS/virchow_wilds_preprocessed_run_01'
RUN_DIR_PCAM_TRAINED  = '/content/drive/MyDrive/GP_ECG_RUNS/virchow_macenko_bench_run_01'

# Preprocessed dataset directories (must contain test_x.h5 and test_y.h5)
PREPROCESSED_CAM17_DIR = '/content/drive/MyDrive/GP_ECG_DATA/wilds/preprocessed_macenko_benchmark_style'
PREPROCESSED_PCAM_DIR  = '/content/drive/MyDrive/GP_ECG_DATA/preprocessed_macenko_benchmark_style'

# Root where completed eval folders are synced (Drive — survives reconnect)
EVAL_OUTPUT_ROOT = '/content/drive/MyDrive/GP_ECG_RUNS/evals_cross_domain'

# Fast scratch on Colab VM (wiped on disconnect; results copied to EVAL_OUTPUT_ROOT after each run)
EVAL_LOCAL_ROOT = '/content/evals_local_cross_domain'

# Passed to evaluate_virchow_preprocessed_test_colab.py (lower batch if CUDA OOM)
BATCH_SIZE = 320
NUM_WORKERS = 0

# Optional MC dropout pass (0 = deterministic only; much faster)
MC_SAMPLES = 0

# Optional: paste token here if Virchow2 hub access is gated
HF_TOKEN = ''

import os
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    os.environ['HUGGINGFACE_HUB_TOKEN'] = HF_TOKEN

print('REPO_DIR =', REPO_DIR)
print('RUN_DIR_CAM17_TRAINED =', RUN_DIR_CAM17_TRAINED)
print('RUN_DIR_PCAM_TRAINED  =', RUN_DIR_PCAM_TRAINED)
print('PREPROCESSED_CAM17_DIR =', PREPROCESSED_CAM17_DIR)
print('PREPROCESSED_PCAM_DIR  =', PREPROCESSED_PCAM_DIR)
print('EVAL_OUTPUT_ROOT =', EVAL_OUTPUT_ROOT)
print('EVAL_LOCAL_ROOT =', EVAL_LOCAL_ROOT)
print('BATCH_SIZE =', BATCH_SIZE, ' NUM_WORKERS =', NUM_WORKERS)
print('MC_SAMPLES =', MC_SAMPLES)

In [ ]:
%cd {REPO_DIR}
!python -m pip install --upgrade pip
!pip uninstall -y torch torchvision torchaudio
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install timm h5py tqdm huggingface_hub scikit-learn

In [ ]:
# --- Copy inputs to local SSD with visible progress ---
import os, shutil, time

COPY_TO_LOCAL_SSD = True

def copy_file_with_log(src, dst):
    t0 = time.time()
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    size_gb = os.path.getsize(src) / (1024**3) if os.path.isfile(src) else 0
    print(f"[COPY] {src}  ({size_gb:.2f} GB)")
    shutil.copy2(src, dst)
    dt = time.time() - t0
    print(f"[DONE] {dst}  in {dt:.1f}s")

if COPY_TO_LOCAL_SSD:
    LOCAL_ROOT = "/content/local_eval_inputs"
    os.makedirs(LOCAL_ROOT, exist_ok=True)

    LOCAL_RUN_DIR_CAM17 = os.path.join(LOCAL_ROOT, "run_cam17_trained")
    LOCAL_RUN_DIR_PCAM  = os.path.join(LOCAL_ROOT, "run_pcam_trained")
    LOCAL_PRE_CAM17     = os.path.join(LOCAL_ROOT, "pre_cam17")
    LOCAL_PRE_PCAM      = os.path.join(LOCAL_ROOT, "pre_pcam")

    # Clean old copies
    for p in [LOCAL_RUN_DIR_CAM17, LOCAL_RUN_DIR_PCAM, LOCAL_PRE_CAM17, LOCAL_PRE_PCAM]:
        if os.path.exists(p):
            shutil.rmtree(p)

    # Copy only required run files (faster than copying full run folders)
    run_needed = ["model_best.pt", "run_config.json", "temperature_fit.json"]
    for src_run, dst_run in [
        (RUN_DIR_CAM17_TRAINED, LOCAL_RUN_DIR_CAM17),
        (RUN_DIR_PCAM_TRAINED,  LOCAL_RUN_DIR_PCAM),
    ]:
        os.makedirs(dst_run, exist_ok=True)
        for fn in run_needed:
            src = os.path.join(src_run, fn)
            if not os.path.isfile(src):
                raise FileNotFoundError(f"Missing run file: {src}")
            copy_file_with_log(src, os.path.join(dst_run, fn))

    # Copy test H5 files
    for src_pre, dst_pre in [
        (PREPROCESSED_CAM17_DIR, LOCAL_PRE_CAM17),
        (PREPROCESSED_PCAM_DIR,  LOCAL_PRE_PCAM),
    ]:
        for fn in ("test_x.h5", "test_y.h5"):
            src = os.path.join(src_pre, fn)
            if not os.path.isfile(src):
                raise FileNotFoundError(f"Missing test file: {src}")
            copy_file_with_log(src, os.path.join(dst_pre, fn))

    # Re-point variables
    RUN_DIR_CAM17_TRAINED = LOCAL_RUN_DIR_CAM17
    RUN_DIR_PCAM_TRAINED  = LOCAL_RUN_DIR_PCAM
    PREPROCESSED_CAM17_DIR = LOCAL_PRE_CAM17
    PREPROCESSED_PCAM_DIR  = LOCAL_PRE_PCAM

    print("\nUsing local SSD copies:")
    print(" RUN_DIR_CAM17_TRAINED =", RUN_DIR_CAM17_TRAINED)
    print(" RUN_DIR_PCAM_TRAINED  =", RUN_DIR_PCAM_TRAINED)
    print(" PREPROCESSED_CAM17_DIR =", PREPROCESSED_CAM17_DIR)
    print(" PREPROCESSED_PCAM_DIR  =", PREPROCESSED_PCAM_DIR)
else:
    print("Using Drive paths directly.")

In [ ]:
import os
required = [
    RUN_DIR_CAM17_TRAINED,
    RUN_DIR_PCAM_TRAINED,
    PREPROCESSED_CAM17_DIR,
    PREPROCESSED_PCAM_DIR,
]
for p in required:
    print(('OK ' if os.path.isdir(p) else 'MISSING '), p)

for p in [RUN_DIR_CAM17_TRAINED, RUN_DIR_PCAM_TRAINED]:
    print('\nRun files in', p)
    for fn in ('model_best.pt', 'run_config.json', 'temperature_fit.json'):
        q = os.path.join(p, fn)
        print(('  OK ' if os.path.isfile(q) else '  MISSING '), q)

for p in [PREPROCESSED_CAM17_DIR, PREPROCESSED_PCAM_DIR]:
    print('\nTest files in', p)
    for fn in ('test_x.h5', 'test_y.h5'):
        q = os.path.join(p, fn)
        print(('  OK ' if os.path.isfile(q) else '  MISSING '), q)

In [ ]:
import os
import shutil
import subprocess
import sys


def _colab_python_exe():
    """Prefer PATH `python` so subprocess matches `!python` / beginner notebook (not always sys.executable)."""
    for cand in (shutil.which('python'), shutil.which('python3')):
        if cand:
            return cand
    return sys.executable


def _run_eval_streaming(cmd, cwd, env):
    """Stream child stdout into this notebook (Colab often buffers inherited subprocess FD until exit)."""
    proc = subprocess.Popen(
        cmd,
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end='', flush=True)
    rc = proc.wait()
    if rc != 0:
        raise subprocess.CalledProcessError(rc, cmd)


def prepare_eval_run_dir(src_run_dir: str, dst_eval_dir: str):
    os.makedirs(dst_eval_dir, exist_ok=True)
    needed = ['model_best.pt', 'run_config.json', 'temperature_fit.json']
    optional = ['checkpoint_last.pt', 'checkpoint.pt', 'metrics_final.json', 'metrics_final_detailed.json']

    for fn in needed:
        s = os.path.join(src_run_dir, fn)
        if not os.path.isfile(s):
            raise FileNotFoundError(f'Missing required file in run dir: {s}')
        shutil.copy2(s, os.path.join(dst_eval_dir, fn))

    for fn in optional:
        s = os.path.join(src_run_dir, fn)
        if os.path.isfile(s):
            shutil.copy2(s, os.path.join(dst_eval_dir, fn))


def sync_eval_to_drive(local_dir: str, drive_dir: str):
    """Copy a finished eval folder to Drive so reconnects can skip completed tests."""
    os.makedirs(drive_dir, exist_ok=True)
    for root, _, files in os.walk(local_dir):
        rel = os.path.relpath(root, local_dir)
        dst_root = os.path.join(drive_dir, rel) if rel != '.' else drive_dir
        os.makedirs(dst_root, exist_ok=True)
        for fn in files:
            shutil.copy2(os.path.join(root, fn), os.path.join(dst_root, fn))
    print('[synced]', local_dir, '->', drive_dir)


def run_one_eval(
    name: str,
    src_run_dir: str,
    preprocessed_dir: str,
    mc_samples: int = 0,
    batch_size: int = None,
    num_workers: int = None,
    skip_if_done: bool = True,
):
    """Evaluate on local SSD, then sync to EVAL_OUTPUT_ROOT on Drive."""
    batch_size = BATCH_SIZE if batch_size is None else batch_size
    num_workers = NUM_WORKERS if num_workers is None else num_workers

    drive_dst = os.path.join(EVAL_OUTPUT_ROOT, name)
    local_dst = os.path.join(EVAL_LOCAL_ROOT, name)
    marker_drive = os.path.join(drive_dst, 'test_metrics.json')
    marker_local = os.path.join(local_dst, 'test_metrics.json')

    os.makedirs(EVAL_OUTPUT_ROOT, exist_ok=True)
    os.makedirs(EVAL_LOCAL_ROOT, exist_ok=True)

    if skip_if_done and os.path.isfile(marker_drive):
        print(f'SKIP (already complete on Drive): {name}')
        return drive_dst
    if skip_if_done and os.path.isfile(marker_local):
        print(f'SKIP (local done, syncing): {name}')
        sync_eval_to_drive(local_dst, drive_dst)
        return drive_dst

    prepare_eval_run_dir(src_run_dir, local_dst)

    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'

    py_exe = _colab_python_exe()
    cmd = [
        py_exe,
        '-u',
        'scripts/evaluate_virchow_preprocessed_test_colab.py',
        '--preprocessed-dir',
        preprocessed_dir,
        '--run-dir',
        local_dst,
        '--batch-size',
        str(batch_size),
        '--num-workers',
        str(num_workers),
    ]
    if mc_samples > 0:
        cmd += ['--mc-samples', str(mc_samples)]

    print('\n' + '=' * 80)
    print('Running:', name)
    print('Run source:', src_run_dir)
    print('Test data :', preprocessed_dir)
    print('Local dir :', local_dst)
    print('Drive sync:', drive_dst)
    print('Python exe:', py_exe, '(same idea as beginner `!python`)')
    print('Command   :', ' '.join(cmd))
    print('=' * 80)

    _run_eval_streaming(cmd, cwd=REPO_DIR, env=env)

    sync_eval_to_drive(local_dst, drive_dst)
    return drive_dst

### Test 1 — CAMELYON17-trained → CAMELYON17 test

Run this cell alone; if Colab disconnects, rerun setup cells above then skip completed tests via Drive checkpoint.

In [ ]:
run_one_eval(
    name='cam17_trained_on_cam17_test',
    src_run_dir=RUN_DIR_CAM17_TRAINED,
    preprocessed_dir=PREPROCESSED_CAM17_DIR,
    mc_samples=MC_SAMPLES,
)

### Test 2 — CAMELYON17-trained → PCam test

In [ ]:
run_one_eval(
    name='cam17_trained_on_pcam_test',
    src_run_dir=RUN_DIR_CAM17_TRAINED,
    preprocessed_dir=PREPROCESSED_PCAM_DIR,
    mc_samples=MC_SAMPLES,
)

### Test 3 — PCam-trained → CAMELYON17 test

In [ ]:
run_one_eval(
    name='pcam_trained_on_cam17_test',
    src_run_dir=RUN_DIR_PCAM_TRAINED,
    preprocessed_dir=PREPROCESSED_CAM17_DIR,
    mc_samples=MC_SAMPLES,
)

### Test 4 — PCam-trained → PCam test

In [ ]:
run_one_eval(
    name='pcam_trained_on_pcam_test',
    src_run_dir=RUN_DIR_PCAM_TRAINED,
    preprocessed_dir=PREPROCESSED_PCAM_DIR,
    mc_samples=MC_SAMPLES,
)

In [ ]:
import os
import json

def show_compact_metrics(eval_dir):
    p = os.path.join(eval_dir, 'test_metrics.json')
    if not os.path.isfile(p):
        print('MISSING', p)
        return
    with open(p, 'r', encoding='utf-8') as f:
        d = json.load(f)
    print('\n', eval_dir)
    for k in ['n_test', 'accuracy_raw', 'roc_auc_raw', 'pr_auc_raw', 'brier_raw', 'ece_raw']:
        print(f'  {k}: {d.get(k)}')

for sub in [
    'cam17_trained_on_cam17_test',
    'cam17_trained_on_pcam_test',
    'pcam_trained_on_cam17_test',
    'pcam_trained_on_pcam_test',
]:
    show_compact_metrics(os.path.join(EVAL_OUTPUT_ROOT, sub))